[![Mở trong Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/trungnguyenvn/mlsysbook-vi-notebooks/blob/main/vol1-ch15-responsible-engr/03-tien-carbon-va-dau-vet-du-lieu.ipynb)

# Cái giá ngoài độ chính xác: tiền, carbon, và dấu vết dữ liệu

Hai sổ tay trước đo cái giá mà một chỉ số tổng hợp che đi, ở phía con người. Chương 15 còn chỉ ra hai
loại chi phí nữa mà bảng điểm thông thường không ghi. Loại thứ nhất nằm trên hoá đơn và trong khí
quyển: điện năng, giờ GPU, phát thải carbon, cộng dồn qua cả vòng đời của hệ thống. Loại thứ hai nằm
trong chính dữ liệu: một bản ghi đã vào huấn luyện để lại dấu vết ở đặc trưng, ở mô hình, và có thể
cả trong trọng số.

**Bạn sẽ làm:**
1. đặt bốn mô hình của bảng 12 lên bốn ngân sách của bảng 11, và tính năng lượng mỗi lần suy luận;
2. quy đổi một giờ GPU ra carbon (phép tính nhẩm 1.3), rồi dựng lại ba sổ cái tổng chi phí sở hữu
   (TCO) của bảng 13 đến 16, từng dòng một;
3. đo trên máy này xem gom batch đổi nhu cầu phục vụ của mỗi truy vấn ra sao, rồi quét hai giả định
   mà sổ cái của sách cố định: hệ số gom batch và mức sử dụng;
4. lặp lại phép tính nhẩm 1.4 cho GPT-3, quét cường độ carbon và PUE, và thử dời một việc huấn luyện
   sang giờ lưới điện sạch hơn;
5. trả lời một yêu cầu xoá dữ liệu bằng đồ thị phả hệ, đo một tấn công suy luận thành viên trước và
   sau khi huấn luyện lại, và mô phỏng cách quyền riêng tư vi sai tiêu ngân sách qua nhiều lần công bố.

In [ ]:
#@title Chuẩn bị: thư viện, hạt giống và hai hàm đối chiếu với sách { display-mode: "form" }
# Chạy ô này trước. Nó không cài thêm gì: mọi thư viện dùng ở đây đều có sẵn trên Colab.
import math, os, random, re, sys, time

import matplotlib.pyplot as plt
import numpy as np
import torch

# Số luồng CPU. 0 để thư viện tự chọn (trên Colab là 2).
NB_THREADS = int(os.environ.get("NB_THREADS", "0"))
if NB_THREADS:
    torch.set_num_threads(NB_THREADS)
    try:
        torch.set_num_interop_threads(NB_THREADS)
    except RuntimeError:  # chỉ đặt được một lần mỗi phiên
        pass
    from threadpoolctl import threadpool_limits
    threadpool_limits(NB_THREADS)

SEED = 42


def dat_hat_giong(seed=SEED):
    """Cùng hạt giống thì cùng kết quả, mỗi lần chạy lại."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


dat_hat_giong()
DATA = os.environ.get("NB_DATA", "data")  # nơi tải dữ liệu về
NB_STRICT = os.environ.get("NB_STRICT") == "1"
plt.rcParams.update({"figure.figsize": (7, 4), "figure.dpi": 110,
                     "axes.grid": True, "grid.alpha": 0.3})

# Đọc một con số theo cách bảng số liệu của bài học viết nó (dấu chấm hay dấu phẩy).
import math
import re

_SUP = str.maketrans("⁰¹²³⁴⁵⁶⁷⁸⁹⁻⁺", "0123456789-+")
_MULT = {"tỷ": 1e9, "tỉ": 1e9, "triệu": 1e6, "nghìn": 1e3, "ngàn": 1e3,
         "K": 1e3, "M": 1e6, "B": 1e9}
_NUM = re.compile(
    r"(?<![\w.,])(-?)(\d+(?:[.,]\d+)*)"                      # sign, digits with separators
    r"(?:\s*[×x·]\s*10\s*(?:\^\{?\s*([-+]?\d+)\s*\}?|([⁻⁺]?[⁰¹²³⁴⁵⁶⁷⁸⁹]+))|[eE]([-+]?\d+)"
    r"|\^\{?\s*([-+]?\d+)\s*\}?|([⁻⁺]?[⁰¹²³⁴⁵⁶⁷⁸⁹]+))?"                  # or a bare power, 10^{-5}
    r"(\s*%|\s*(?:tỷ|tỉ|triệu|nghìn|ngàn)(?!\w)|\s?[KMB](?![A-Za-z]))?")


def norm(text):
    """The outline's markup and typography reduced to plain text, as both sides compare it."""
    text = text.replace("{,}", ",").replace("**", "").replace("`", "")
    text = re.sub(r"[   ]", " ", text)
    text = re.sub(r"−(?=\d)", "-", text)
    text = re.sub(r"[~≈]", "", text)
    return re.sub(r"\s+", " ", text).strip()


def _readings(digits):
    """(value, decimals) for each way the separators in `digits` can be read."""
    out = []
    groups_ok = lambda parts: all(len(p) == 3 for p in parts[1:])
    for thou, dec in ((",", "."), (".", ",")):
        if thou in digits and dec in digits:
            head, _, tail = digits.rpartition(dec)
            if dec in head or not tail.isdigit() or not groups_ok(head.split(thou)):
                continue
            out.append((float(head.replace(thou, "") + "." + tail), len(tail)))
    for sep in (",", "."):
        other = "." if sep == "," else ","
        if sep in digits and other not in digits:
            parts = digits.split(sep)
            if groups_ok(parts):
                out.append((float("".join(parts)), 0))                    # thousands
            if len(parts) == 2:
                out.append((float(parts[0] + "." + parts[1]), len(parts[1])))  # decimal
    if "." not in digits and "," not in digits:
        out.append((float(digits), 0))
    return out


def candidates(text):
    """Every (value, half_unit) the numbers in `text` can mean.

    half_unit is half of the last printed digit, scaled like the value, so `437,5 KB`
    accepts 437.544. Each number is offered as written and, when a multiplier or `%`
    follows it, scaled by that too, so `8,5 tỉ` cites 8.5 or 8.5e9 and `20 %` cites 20 or 0.2.
    """
    out = []
    for m in _NUM.finditer(norm(text)):
        sign, digits, exp_caret, exp_sup, exp_e, pow_caret, pow_sup, suffix = m.groups()
        exp = exp_caret or (exp_sup.translate(_SUP) if exp_sup else None) or exp_e
        power = pow_caret or (pow_sup.translate(_SUP) if pow_sup else None)
        scale10 = 10.0 ** int(exp) if exp else 1.0
        suffix = (suffix or "").strip()
        extra = 0.01 if suffix == "%" else _MULT.get(suffix)
        for value, decimals in _readings(digits):
            if power:
                # 10^{-5}, 2³²: an exact value, so it is matched to 1 part in 10⁹; a looser
                # comparison needs an explicit tol.
                v = value ** int(power)
                half = abs(v) * 1e-9
            else:
                v = value * scale10
                half = 0.5 * 10.0 ** -decimals * scale10
            v = -v if sign else v
            out.append((v, half))
            if extra:
                out.append((v * extra, half * extra))
    return out


def match(value, text):
    """The (value, half_unit) reading of `text` that `value` equals, or None."""
    for v, half in candidates(text):
        if math.isclose(value, v, rel_tol=1e-9, abs_tol=1e-300):
            return v, half
    return None


def well_anchored(text):
    """False for a citation too bare to identify one cell, like "1" or "2"."""
    t = norm(text)
    rest = re.sub(r"[\d.,\s\-+]", "", t)
    if rest:
        return True
    sig = re.sub(r"[^\d]", "", t).lstrip("0")
    return len(sig) >= 2


def _so(x):
    """Một con số để in: số nguyên có dấu phẩy ngăn nghìn, số thực giữ 6 chữ số có nghĩa."""
    if isinstance(x, (int, np.integer)) or (isinstance(x, float) and x.is_integer() and abs(x) < 1e15):
        return f"{int(x):,}"
    return f"{x:,.6g}"


def sach(gia_tri, nguon=None, trich=None, tol=None):
    """Một con số của sách, kèm đúng chữ của nó: trong bảng số liệu của bài học (nguon)
    hoặc nguyên văn trong chương (trich)."""
    chu = nguon if nguon is not None else trich
    if match(gia_tri, chu) is None:
        raise ValueError(f"{gia_tri!r} không phải con số ghi trong {chu!r}")
    return gia_tri


def theo_sach(ten, gia_tri, sach, nguon=None, trich=None, tol=None):
    """So một con số tính được với con số của sách. In ✓ khi khớp, ✗ kèm độ lệch khi không."""
    chu = nguon if nguon is not None else trich
    khop = match(sach, chu)
    if khop is None:
        raise ValueError(f"{sach!r} không phải con số ghi trong {chu!r}")
    sai_so = tol if tol is not None else khop[1]
    dung = abs(gia_tri - sach) <= sai_so
    print(f"{'✓' if dung else '✗'} {ten}: tính được {_so(gia_tri)} · sách: {chu}")
    if not dung:
        loi = f"lệch {_so(gia_tri - sach)}, quá sai số cho phép {_so(sai_so)}"
        if NB_STRICT:
            raise AssertionError(f"{ten}: {loi}")
        print(f"   {loi}. Nếu bạn vừa đổi tham số ở trên thì đây là điều được chờ đợi.")
    # None, not the bool: a cell ending in theo_sach(...) would otherwise echo True under the ✓.


def do_tren_may(ten, gia_tri, don_vi=""):
    """Một con số ĐO trên máy đang chạy: mỗi máy một khác, không phải con số của sách."""
    print(f"⏱ {ten}: {_so(gia_tri)} {don_vi} (đo trên máy này)".replace("  ", " "))


print(f"Python {sys.version.split()[0]} · PyTorch {torch.__version__} · NumPy {np.__version__}"
      f" · {torch.get_num_threads()} luồng CPU")

In [ ]:
import torchvision
from sklearn.metrics import roc_auc_score

MAU = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#8e5cc4"]  # thứ tự màu cố định
XAM = "0.45"
GIAY_MOI_GIO, NGAY_MOI_NAM = 3600, 365  # hằng số lịch, không phải số của sách
print("sẵn sàng")

## 1. Ngân sách ở biên: bảng 11 và 12

Ở thiết bị biên, hiệu quả không phải một sở thích tinh chỉnh mà là một giới hạn vật lý. Bảng 11 của
chương đặt bốn bối cảnh triển khai, mỗi bối cảnh một ngân sách công suất và một yêu cầu độ trễ, đều là
giả định của kịch bản. Bảng 12 cho bốn hồ sơ mô hình minh hoạ: công suất khi suy luận và độ trễ.

Một mô hình vừa một bối cảnh khi cả hai điều kiện cùng đạt: công suất không quá ngân sách, độ trễ
không quá yêu cầu. Bảng 12 tự ghi hai cột "vừa điện thoại?" và "vừa IoT?". Ta tính lại cả bốn cột,
rồi kiểm hai cột sách đã ghi.

In [ ]:
BOI_CANH = {  # bảng 11: (ngân sách công suất W, yêu cầu độ trễ s)
    "điện thoại": (sach(5, nguon="5 W 100 ms · 100 mW 1 giây · 1 W 33 ms · 500 mW 500 ms"),
                   sach(100, nguon="5 W 100 ms · 100 mW 1 giây · 1 W 33 ms · 500 mW 500 ms") / 1000),
    "cảm biến IoT": (sach(100, nguon="5 W 100 ms · 100 mW 1 giây · 1 W 33 ms · 500 mW 500 ms") / 1000,
                     sach(1, nguon="5 W 100 ms · 100 mW 1 giây · 1 W 33 ms · 500 mW 500 ms")),
    "camera nhúng": (sach(1, nguon="5 W 100 ms · 100 mW 1 giây · 1 W 33 ms · 500 mW 500 ms"),
                     sach(33, nguon="5 W 100 ms · 100 mW 1 giây · 1 W 33 ms · 500 mW 500 ms") / 1000),
    "thiết bị đeo": (sach(500, nguon="5 W 100 ms · 100 mW 1 giây · 1 W 33 ms · 500 mW 500 ms") / 1000,
                     sach(500, nguon="5 W 100 ms · 100 mW 1 giây · 1 W 33 ms · 500 mW 500 ms") / 1000),
}
MO_HINH = {  # bảng 12: (công suất khi suy luận W, độ trễ s)
    "MobileNetV2": (sach(1.2, nguon="3.5M 1.2 W 40 ms · 5.3M 1.8 W 65 ms · 25.6M 4.5 W 180 ms · 200K 50 mW 200 ms"),
                    sach(40, nguon="3.5M 1.2 W 40 ms · 5.3M 1.8 W 65 ms · 25.6M 4.5 W 180 ms · 200K 50 mW 200 ms") / 1000),
    "EfficientNet-B0": (sach(1.8, nguon="3.5M 1.2 W 40 ms · 5.3M 1.8 W 65 ms · 25.6M 4.5 W 180 ms · 200K 50 mW 200 ms"),
                        sach(65, nguon="3.5M 1.2 W 40 ms · 5.3M 1.8 W 65 ms · 25.6M 4.5 W 180 ms · 200K 50 mW 200 ms") / 1000),
    "ResNet-50": (sach(4.5, nguon="3.5M 1.2 W 40 ms · 5.3M 1.8 W 65 ms · 25.6M 4.5 W 180 ms · 200K 50 mW 200 ms"),
                  sach(180, nguon="3.5M 1.2 W 40 ms · 5.3M 1.8 W 65 ms · 25.6M 4.5 W 180 ms · 200K 50 mW 200 ms") / 1000),
    "TinyML": (sach(50, nguon="3.5M 1.2 W 40 ms · 5.3M 1.8 W 65 ms · 25.6M 4.5 W 180 ms · 200K 50 mW 200 ms") / 1000,
               sach(200, nguon="3.5M 1.2 W 40 ms · 5.3M 1.8 W 65 ms · 25.6M 4.5 W 180 ms · 200K 50 mW 200 ms") / 1000),
}
SACH_GHI = {  # hai cột "Fits Smartphone?" và "Fits IoT?" của bảng 12
    "MobileNetV2": (True, False), "EfficientNet-B0": (True, False),
    "ResNet-50": (False, False), "TinyML": (False, True),
}


def vua(mo_hinh, boi_canh):
    """(vừa hay không, danh sách điều kiện bị vượt)."""
    p, t = MO_HINH[mo_hinh]
    ngan_sach, yeu_cau = BOI_CANH[boi_canh]
    vuot = [ten for ten, qua in (("công suất", p > ngan_sach), ("độ trễ", t > yeu_cau)) if qua]
    return not vuot, vuot


print(f"{'':>16} | " + " | ".join(f"{b:>14}" for b in BOI_CANH))
for m in MO_HINH:
    o = []
    for b in BOI_CANH:
        ok, vuot = vua(m, b)
        o.append(f"{'vừa' if ok else 'quá ' + ' và '.join(vuot):>14}")
    print(f"{m:>16} | " + " | ".join(o))
for m, (dt, iot) in SACH_GHI.items():
    assert vua(m, "điện thoại")[0] == dt and vua(m, "cảm biến IoT")[0] == iot, m
print("hai cột của sách: khớp cả 8 ô")

Hai cột khớp sách, và bảng tính ra còn cho thấy điều cột "có, không" giấu đi: mỗi lần "không" có một
lý do riêng. ResNet-50 không vừa điện thoại vì độ trễ chứ không phải vì công suất. TinyML không vừa
điện thoại cũng vì độ trễ, dù nó tiêu thụ ít điện nhất. Chọn mô hình theo một trục duy nhất sẽ chọn
sai.

Sách tính thêm hai biên cho thiết bị đeo: TinyML còn dư công suất, còn MobileNetV2 vượt ngân sách.

In [ ]:
P_DEO = BOI_CANH["thiết bị đeo"][0]
theo_sach("biên công suất của TinyML trên thiết bị đeo (lần)", P_DEO / MO_HINH["TinyML"][0],
          sach=10, nguon="10 lần · 2.4 lần")
theo_sach("MobileNetV2 vượt ngân sách thiết bị đeo (lần)", MO_HINH["MobileNetV2"][0] / P_DEO,
          sach=2.4, nguon="10 lần · 2.4 lần")

### Dự đoán

Ngân sách công suất là giới hạn tức thời: mạch điện, nhiệt, dòng tối đa của pin. Pin còn đặt một giới
hạn thứ hai, về năng lượng: thiết bị đeo của kịch bản phải chạy 24 giờ với ngân sách 500 mW, tức một
lượng điện năng cố định mỗi ngày. Giả sử (giả định của sổ tay) thiết bị chỉ cần một lần suy luận mỗi
giây, và giữa hai lần thì gần như không tốn điện. MobileNetV2, mô hình vượt ngân sách công suất 2.4 lần,
có vượt ngân sách năng lượng một ngày không?

In [ ]:
NGAN_SACH_NGAY = (sach(500, nguon="500 mW · 24 giờ") / 1000) * sach(24, nguon="500 mW · 24 giờ")  # Wh
TAN_SUAT = 1.0  # giả định: một lần suy luận mỗi giây
print(f"ngân sách năng lượng một ngày của thiết bị đeo: {NGAN_SACH_NGAY:.1f} Wh")
NANG_LUONG = {}
for m, (p, t) in MO_HINH.items():
    mj = p * t * 1000                       # năng lượng mỗi lần suy luận = công suất × thời gian
    lien_tuc = p * 24                       # Wh nếu chạy nối tiếp không nghỉ suốt 24 giờ
    theo_nhip = mj / 1000 * TAN_SUAT * 86400 / 3600  # Wh với một lần mỗi giây
    NANG_LUONG[m] = mj
    print(f"{m:>16}: {mj:6.1f} mJ mỗi lần · chạy không nghỉ {lien_tuc:6.1f} Wh/ngày · "
          f"một lần mỗi giây {theo_nhip:5.2f} Wh/ngày ({theo_nhip / NGAN_SACH_NGAY:6.1%} ngân sách)")

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 3.9), gridspec_kw=dict(width_ratios=[1.35, 1]))
ten_m, ten_b = list(MO_HINH), list(BOI_CANH)
mau_o = {(): MAU[2], ("công suất",): MAU[3], ("độ trễ",): MAU[0], ("công suất", "độ trễ"): MAU[1]}
for i, m in enumerate(ten_m):
    for j, b in enumerate(ten_b):
        ok, vuot = vua(m, b)
        ax1.add_patch(plt.Rectangle((j - 0.47, i - 0.45), 0.94, 0.9, color=mau_o[tuple(vuot)], alpha=0.85))
        ax1.text(j, i, "vừa" if ok else "quá\n" + "\nvà ".join(vuot), ha="center", va="center", fontsize=8,
                 color="black" if tuple(vuot) == ("công suất",) else "white")
ax1.set_xlim(-0.5, len(ten_b) - 0.5)
ax1.set_ylim(len(ten_m) - 0.5, -0.5)
ax1.set_xticks(range(len(ten_b)))
ax1.set_xticklabels([f"{b}\n{BOI_CANH[b][0]:g} W · {BOI_CANH[b][1] * 1000:g} ms" for b in ten_b], fontsize=8)
ax1.set_yticks(range(len(ten_m)))
ax1.set_yticklabels(ten_m)
ax1.tick_params(length=0)
for s in ax1.spines.values():
    s.set_visible(False)
ax1.grid(False)
ax1.set_title("Mô hình nào vừa ngân sách nào (bảng 11 × bảng 12)", fontsize=10)

y = np.arange(len(ten_m))
ax2.barh(y, [NANG_LUONG[m] for m in ten_m], color=MAU[0], height=0.6)
for yi, m in zip(y, ten_m):
    ax2.text(NANG_LUONG[m] * 1.15, yi, f"{NANG_LUONG[m]:.0f} mJ", va="center", fontsize=8)
ax2.set_xscale("log")
ax2.set_xlim(3, 5000)
ax2.set_yticks(y)
ax2.set_yticklabels(ten_m)
ax2.invert_yaxis()
ax2.set_xlabel("năng lượng mỗi lần suy luận (mJ, thang log)")
ax2.set_title("Công suất × độ trễ", fontsize=10)
plt.tight_layout()
plt.show()

Năng lượng mỗi lần suy luận chênh nhau gần hai bậc giữa TinyML và ResNet-50, và đó là con số nhân lên
theo số lần chạy. Với nhịp một lần mỗi giây (giả định), MobileNetV2 chỉ tiêu một phần nhỏ năng lượng
của ngày, dù nó vượt ngân sách công suất: hai giới hạn là hai câu hỏi khác nhau, và một thiết kế phải
qua cả hai. Còn chạy không nghỉ thì năng lượng một ngày chỉ là công suất nhân 24 giờ, nên mô hình nào
vượt ngân sách công suất cũng vượt luôn ngân sách năng lượng.

## 2. Giờ GPU thành tiền và carbon: ba sổ cái

### Phép tính nhẩm 1.3: một giờ GPU nặng bao nhiêu carbon

Để sổ cái ghi được carbon bên cạnh đô la, cần một hệ số quy đổi. Phương trình 2 của chương: lượng
carbon bằng năng lượng (kWh) nhân cường độ carbon của lưới điện (kg mỗi kWh). Kịch bản lấy công suất
một GPU là 400 W và cường độ carbon làm tròn là 0.4 kg mỗi kWh.

In [ ]:
CONG_SUAT_GPU = sach(400, nguon="400 W · 0.4 kg mỗi kWh · 0.16 kg") / 1000  # kW
CUONG_DO_CARBON = sach(0.4, nguon="400 W · 0.4 kg mỗi kWh · 0.16 kg")      # kg CO2 mỗi kWh
KG_MOI_GIO_GPU = CONG_SUAT_GPU * 1 * CUONG_DO_CARBON
theo_sach("carbon của một giờ GPU (kg CO2)", KG_MOI_GIO_GPU, sach=0.16, nguon="400 W · 0.4 kg mỗi kWh · 0.16 kg")

### Sổ cái huấn luyện: bảng 13

Hệ thống gợi ý của kịch bản phục vụ 10 triệu người dùng mỗi ngày. Một chu kỳ huấn luyện gồm ba khoản
giờ GPU ở giá 4 đô la mỗi giờ: chuẩn bị dữ liệu, tìm siêu tham số, và lần huấn luyện cuối. Mô hình
được huấn luyện lại mỗi quý, trong ba năm vận hành.

In [ ]:
GIA_GIO_HUAN_LUYEN = sach(4, nguon="10 triệu · 100 · 4 · 400 · 500 · 2,000 · 200 · 800 · 3,200")  # $/giờ GPU
GIO_HUAN_LUYEN = {
    "chuẩn bị dữ liệu": sach(100, nguon="10 triệu · 100 · 4 · 400 · 500 · 2,000 · 200 · 800 · 3,200"),
    "tìm siêu tham số": sach(500, nguon="10 triệu · 100 · 4 · 400 · 500 · 2,000 · 200 · 800 · 3,200"),
    "huấn luyện cuối": sach(200, nguon="10 triệu · 100 · 4 · 400 · 500 · 2,000 · 200 · 800 · 3,200"),
}
tien_chu_ky = kg_chu_ky = 0.0
for ten, gio in GIO_HUAN_LUYEN.items():
    tien, kg = gio * GIA_GIO_HUAN_LUYEN, gio * KG_MOI_GIO_GPU
    tien_chu_ky += tien
    kg_chu_ky += kg
    print(f"{ten:>17}: {gio:4.0f} giờ GPU → ${tien:,.0f} · {kg:.0f} kg CO2")
theo_sach("chuẩn bị dữ liệu ($)", GIO_HUAN_LUYEN["chuẩn bị dữ liệu"] * GIA_GIO_HUAN_LUYEN, sach=400,
          nguon="10 triệu · 100 · 4 · 400 · 500 · 2,000 · 200 · 800 · 3,200")
theo_sach("tìm siêu tham số ($)", GIO_HUAN_LUYEN["tìm siêu tham số"] * GIA_GIO_HUAN_LUYEN, sach=2000,
          nguon="10 triệu · 100 · 4 · 400 · 500 · 2,000 · 200 · 800 · 3,200")
theo_sach("huấn luyện cuối ($)", GIO_HUAN_LUYEN["huấn luyện cuối"] * GIA_GIO_HUAN_LUYEN, sach=800,
          nguon="10 triệu · 100 · 4 · 400 · 500 · 2,000 · 200 · 800 · 3,200")
theo_sach("một chu kỳ huấn luyện ($)", tien_chu_ky, sach=3200,
          nguon="10 triệu · 100 · 4 · 400 · 500 · 2,000 · 200 · 800 · 3,200")
theo_sach("carbon, chuẩn bị dữ liệu (kg)", GIO_HUAN_LUYEN["chuẩn bị dữ liệu"] * KG_MOI_GIO_GPU, sach=16,
          nguon="16 · 80 · 32 · 128 kg · 12 chu kỳ · 38,400 · 1,536 kg")
theo_sach("carbon, tìm siêu tham số (kg)", GIO_HUAN_LUYEN["tìm siêu tham số"] * KG_MOI_GIO_GPU, sach=80,
          nguon="16 · 80 · 32 · 128 kg · 12 chu kỳ · 38,400 · 1,536 kg")
theo_sach("carbon, huấn luyện cuối (kg)", GIO_HUAN_LUYEN["huấn luyện cuối"] * KG_MOI_GIO_GPU, sach=32,
          nguon="16 · 80 · 32 · 128 kg · 12 chu kỳ · 38,400 · 1,536 kg")
theo_sach("carbon một chu kỳ (kg)", kg_chu_ky, sach=128, nguon="16 · 80 · 32 · 128 kg · 12 chu kỳ · 38,400 · 1,536 kg")

CHU_KY_MOI_NAM = sach(4, trich="4/year")    # huấn luyện lại mỗi quý
SO_NAM = sach(3, trich="3 years = 12")      # thời gian vận hành
CHU_KY = CHU_KY_MOI_NAM * SO_NAM
HL_TIEN, HL_KG = tien_chu_ky * CHU_KY, kg_chu_ky * CHU_KY
theo_sach("số chu kỳ trong ba năm", CHU_KY, sach=12, nguon="16 · 80 · 32 · 128 kg · 12 chu kỳ · 38,400 · 1,536 kg")
theo_sach("huấn luyện ba năm ($)", HL_TIEN, sach=38400, nguon="16 · 80 · 32 · 128 kg · 12 chu kỳ · 38,400 · 1,536 kg")
theo_sach("carbon huấn luyện ba năm (kg)", HL_KG, sach=1536, nguon="16 · 80 · 32 · 128 kg · 12 chu kỳ · 38,400 · 1,536 kg")

### Sổ cái suy luận: bảng 14

Bảng 14 đi theo một chuỗi quy đổi: truy vấn mỗi ngày thành giây GPU, giây GPU thành giờ GPU, giờ GPU
thành đô la và carbon. Giả định quyết định của chuỗi là mỗi truy vấn chiếm trọn 10 ms của một GPU
dành riêng, không gom batch và không chạy chồng. Sổ tay giữ các bước dưới dạng hàm, để phần 3 đổi
được giả định ấy.

In [ ]:
NGUOI_DUNG = sach(10e6, nguon="10 triệu · 100 · 4 · 400 · 500 · 2,000 · 200 · 800 · 3,200")
GOI_Y_MOI_NGUOI = sach(20, nguon="20 · 200 triệu · 10 ms · 2 triệu giây · 556 giờ GPU · 23.1 GPU · 2.5 đô la · 506,944 · 1.52 triệu")
NHU_CAU_PHUC_VU = sach(10, nguon="20 · 200 triệu · 10 ms · 2 triệu giây · 556 giờ GPU · 23.1 GPU · 2.5 đô la · 506,944 · 1.52 triệu") / 1000  # giây GPU
GIA_GIO_SUY_LUAN = sach(2.5, nguon="20 · 200 triệu · 10 ms · 2 triệu giây · 556 giờ GPU · 23.1 GPU · 2.5 đô la · 506,944 · 1.52 triệu")


def gio_gpu_moi_ngay(nhu_cau=NHU_CAU_PHUC_VU, he_so_gom=1.0, muc_su_dung=1.0):
    """Giờ GPU phải trả mỗi ngày: truy vấn × nhu cầu phục vụ ÷ hệ số gom batch, chia mức sử dụng."""
    return NGUOI_DUNG * GOI_Y_MOI_NGUOI * nhu_cau / he_so_gom / GIAY_MOI_GIO / muc_su_dung


truy_van = NGUOI_DUNG * GOI_Y_MOI_NGUOI
gio_ngay = gio_gpu_moi_ngay()
SL_NAM = gio_ngay * NGAY_MOI_NAM * GIA_GIO_SUY_LUAN
SL_TIEN = SL_NAM * SO_NAM
SL_KG = gio_ngay * KG_MOI_GIO_GPU * NGAY_MOI_NAM * SO_NAM
theo_sach("truy vấn mỗi ngày", truy_van, sach=200e6, nguon="20 · 200 triệu · 10 ms · 2 triệu giây · 556 giờ GPU · 23.1 GPU · 2.5 đô la · 506,944 · 1.52 triệu")
theo_sach("giây GPU mỗi ngày", truy_van * NHU_CAU_PHUC_VU, sach=2e6, nguon="20 · 200 triệu · 10 ms · 2 triệu giây · 556 giờ GPU · 23.1 GPU · 2.5 đô la · 506,944 · 1.52 triệu")
theo_sach("giờ GPU mỗi ngày", gio_ngay, sach=556, nguon="20 · 200 triệu · 10 ms · 2 triệu giây · 556 giờ GPU · 23.1 GPU · 2.5 đô la · 506,944 · 1.52 triệu")
theo_sach("số GPU chạy liên tục", gio_ngay / 24, sach=23.1, nguon="20 · 200 triệu · 10 ms · 2 triệu giây · 556 giờ GPU · 23.1 GPU · 2.5 đô la · 506,944 · 1.52 triệu")
theo_sach("chi phí GPU mỗi năm ($)", SL_NAM, sach=506944, nguon="20 · 200 triệu · 10 ms · 2 triệu giây · 556 giờ GPU · 23.1 GPU · 2.5 đô la · 506,944 · 1.52 triệu")
theo_sach("suy luận ba năm ($)", SL_TIEN, sach=1.52e6, nguon="20 · 200 triệu · 10 ms · 2 triệu giây · 556 giờ GPU · 23.1 GPU · 2.5 đô la · 506,944 · 1.52 triệu")
theo_sach("carbon suy luận mỗi ngày (kg)", gio_ngay * KG_MOI_GIO_GPU, sach=88.9, nguon="88.9 kg · 32,444.4 kg · 97,333.3 kg")
theo_sach("carbon suy luận mỗi năm (kg)", gio_ngay * KG_MOI_GIO_GPU * NGAY_MOI_NAM, sach=32444.4, nguon="88.9 kg · 32,444.4 kg · 97,333.3 kg")
theo_sach("carbon suy luận ba năm (kg)", SL_KG, sach=97333.3, nguon="88.9 kg · 32,444.4 kg · 97,333.3 kg")
theo_sach("một chu kỳ huấn luyện, câu mở đầu ($)", tien_chu_ky, sach=3200, nguon="3,200 · 500,000 đô la mỗi năm")
theo_sach("phục vụ mỗi năm, câu mở đầu ($)", SL_NAM, sach=500000, nguon="3,200 · 500,000 đô la mỗi năm", tol=10000)

Mọi dòng khớp bảng 14. Dòng cuối là câu mở đầu của mục TCO, nơi sách làm tròn 506,944 đô la thành
500,000 đô la một năm, nên phép so cho phép một sai số. Một đội thấy 3,200 đô la cho một lần huấn luyện,
rồi sáu tháng sau mới nhận ra tiền phục vụ mỗi năm gấp hơn trăm lần con số ấy.

### Sổ cái vận hành và bảng tổng: bảng 15 và 16

In [ ]:
VAN_HANH_NAM = {  # bảng 15, đô la mỗi năm
    "hạ tầng theo dõi": sach(50, nguon="50 · 150 · 100 · 300 · 20 · 60 · 510 nghìn") * 1000,
    "kỹ sư trực, nửa thời gian": sach(100, nguon="50 · 150 · 100 · 300 · 20 · 60 · 510 nghìn") * 1000,
    "ứng phó sự cố": sach(20, nguon="50 · 150 · 100 · 300 · 20 · 60 · 510 nghìn") * 1000,
}
theo_sach("hạ tầng theo dõi, ba năm (nghìn $)", VAN_HANH_NAM["hạ tầng theo dõi"] * SO_NAM / 1000, sach=150,
          nguon="50 · 150 · 100 · 300 · 20 · 60 · 510 nghìn")
theo_sach("kỹ sư trực, ba năm (nghìn $)", VAN_HANH_NAM["kỹ sư trực, nửa thời gian"] * SO_NAM / 1000, sach=300,
          nguon="50 · 150 · 100 · 300 · 20 · 60 · 510 nghìn")
theo_sach("ứng phó sự cố, ba năm (nghìn $)", VAN_HANH_NAM["ứng phó sự cố"] * SO_NAM / 1000, sach=60,
          nguon="50 · 150 · 100 · 300 · 20 · 60 · 510 nghìn")
VH_TIEN = sum(VAN_HANH_NAM.values()) * SO_NAM
theo_sach("vận hành ba năm ($)", VH_TIEN, sach=510000, nguon="50 · 150 · 100 · 300 · 20 · 60 · 510 nghìn")

TONG = HL_TIEN + SL_TIEN + VH_TIEN
theo_sach("tổng chi phí sở hữu ba năm ($)", TONG, sach=2.07e6, nguon="1.9 · 73.5 · 24.6 phần trăm · 2.07 triệu · 1.5 · 97.3 · khoảng 98.9 tấn")
theo_sach("phần của huấn luyện (%)", 100 * HL_TIEN / TONG, sach=1.9, nguon="1.9 · 73.5 · 24.6 phần trăm · 2.07 triệu · 1.5 · 97.3 · khoảng 98.9 tấn")
theo_sach("phần của suy luận (%)", 100 * SL_TIEN / TONG, sach=73.5, nguon="1.9 · 73.5 · 24.6 phần trăm · 2.07 triệu · 1.5 · 97.3 · khoảng 98.9 tấn")
theo_sach("phần của vận hành (%)", 100 * VH_TIEN / TONG, sach=24.6, nguon="1.9 · 73.5 · 24.6 phần trăm · 2.07 triệu · 1.5 · 97.3 · khoảng 98.9 tấn")
theo_sach("carbon huấn luyện (tấn)", HL_KG / 1000, sach=1.5, nguon="1.9 · 73.5 · 24.6 phần trăm · 2.07 triệu · 1.5 · 97.3 · khoảng 98.9 tấn")
theo_sach("carbon suy luận (tấn)", SL_KG / 1000, sach=97.3, nguon="1.9 · 73.5 · 24.6 phần trăm · 2.07 triệu · 1.5 · 97.3 · khoảng 98.9 tấn")
theo_sach("carbon mô hình hoá của máy tăng tốc (tấn)", (HL_KG + SL_KG) / 1000, sach=98.9,
          nguon="1.9 · 73.5 · 24.6 phần trăm · 2.07 triệu · 1.5 · 97.3 · khoảng 98.9 tấn")
theo_sach("chi phí suy luận so với huấn luyện", SL_TIEN / HL_TIEN, sach=40, nguon="40 trên 1 · ba năm · quý")
theo_sach("carbon suy luận so với huấn luyện (lần)", SL_KG / HL_KG, sach=63, nguon="khoảng 63 lần")

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 3.3), gridspec_kw=dict(width_ratios=[1.6, 1]))
phan = [("huấn luyện", HL_TIEN, "0.6"), ("suy luận", SL_TIEN, MAU[1]), ("vận hành", VH_TIEN, "0.78")]
trai = 0.0
for ten, v, mau in phan:
    ax1.barh(0, v / 1e3, left=trai, color=mau, height=0.5)
    nhan = f"{ten}\n${v / 1e3:,.0f} nghìn · {v / TONG:.1%}"
    if v / TONG > 0.1:
        ax1.text(trai + v / 2e3, 0, nhan, ha="center", va="center", fontsize=9)
    else:
        ax1.annotate(nhan, (trai + v / 2e3, 0.25), xytext=(trai + v / 2e3, 0.62), ha="left", va="bottom",
                     fontsize=9, arrowprops=dict(arrowstyle="-", color=XAM, lw=0.8))
    trai += v / 1e3
ax1.set_ylim(-0.5, 1.0)
ax1.set_yticks([])
ax1.set_xlim(0, TONG / 1e3 * 1.02)
ax1.set_axisbelow(True)
ax1.set_xlabel("chi phí ba năm (nghìn $)")
ax1.set_title("Tổng chi phí sở hữu của kịch bản (bảng 16)", fontsize=10)

ax2.bar([0, 1], [HL_KG / 1000, SL_KG / 1000], color=["0.6", MAU[1]], width=0.6)
for x, v in ((0, HL_KG / 1000), (1, SL_KG / 1000)):
    ax2.text(x, v + 2, f"{v:.1f} tấn", ha="center", fontsize=9)
ax2.text(2, 6, "không\nước tính", ha="center", fontsize=9, color=XAM)
ax2.set_xticks([0, 1, 2])
ax2.set_xticklabels(["huấn luyện", "suy luận", "vận hành"])
ax2.set_xlim(-0.6, 2.6)
ax2.set_ylim(0, SL_KG / 1000 * 1.18)
ax2.set_ylabel("CO2 ba năm (tấn)")
ax2.set_title("Carbon mô hình hoá của máy tăng tốc", fontsize=10)
plt.tight_layout()
plt.show()

### Dự đoán

Mục tự kiểm tra thứ tư của chương hỏi: trong kịch bản này, vì sao giảm 20 phần trăm nhu cầu phục vụ
của mỗi truy vấn tiết kiệm nhiều hơn giảm 50 phần trăm thời gian huấn luyện? Đoán trước tỉ số giữa hai
khoản tiết kiệm, rồi chạy ô dưới.

In [ ]:
GIAM_SL = sach(20, nguon="20 phần trăm · 304 nghìn đô la · 19 tấn") / 100
theo_sach("tiết kiệm khi giảm 20 phần trăm nhu cầu mỗi truy vấn ($)", GIAM_SL * SL_TIEN, sach=304000,
          nguon="20 phần trăm · 304 nghìn đô la · 19 tấn")
theo_sach("CO2 mô hình hoá tránh được (tấn)", GIAM_SL * SL_KG / 1000, sach=19,
          nguon="20 phần trăm · 304 nghìn đô la · 19 tấn")
GIAM_HL = sach(50, nguon="20 · 50 phần trăm") / 100
tk_hl = GIAM_HL * HL_TIEN
print(f"giảm {GIAM_HL:.0%} thời gian huấn luyện tiết kiệm ${tk_hl:,.0f} và {GIAM_HL * HL_KG:,.0f} kg CO2 "
      f"(tự tính, sách chỉ đặt câu hỏi)")
print(f"tỉ số giữa hai khoản tiết kiệm: {GIAM_SL * SL_TIEN / tk_hl:.1f} lần")

Câu trả lời nằm ở tỉ lệ 40 trên 1. Một phần trăm nhỏ của khoản lớn vẫn lớn hơn một nửa của khoản nhỏ,
nên ở kịch bản này nỗ lực tối ưu nên dồn vào độ trễ và hiệu quả phục vụ. Sách nhấn mạnh tỉ lệ ấy chỉ
đúng với các giả định của kịch bản: một hệ thống khác có thể bị chi phí huấn luyện, dữ liệu, nhân sự hay
năng lực nhàn rỗi chi phối, và chính phép đo quyết định nên tối ưu chỗ nào. Đây cũng là ngộ nhận cuối
của chương: đo tác động môi trường của huấn luyện mà bỏ quên suy luận.

## 3. Đo trên máy này: gom batch đổi nhu cầu phục vụ

Sổ cái của bảng 14 cố định hai điều mà sách gọi thẳng tên: không gom batch, và mỗi truy vấn chiếm trọn
GPU. Chú thích của bảng viết rằng mức sử dụng, gom batch và chạy đồng thời thật sự sẽ đổi phép quy đổi
này. Ta đo điều đầu tiên trên CPU của máy này, với một mô hình xếp hạng nhỏ dựng ra: chạy cùng một
mạng cho một lô B truy vấn, rồi chia thời gian cho B.

### Dự đoán

Đi từ lô 1 lên lô 512, thời gian cho mỗi truy vấn giảm bao nhiêu lần? Còn thời gian của cả lô, tức độ
trễ mà truy vấn cuối cùng trong lô phải chờ, thì sao?

In [ ]:
torch.manual_seed(SEED)
mo_hinh_xep_hang = torch.nn.Sequential(
    torch.nn.Linear(128, 512), torch.nn.ReLU(), torch.nn.Linear(512, 512), torch.nn.ReLU(), torch.nn.Linear(512, 1),
).eval()
CO_LO = [1, 2, 4, 8, 16, 32, 64, 128, 256, 512]
thoi_gian_lo = {}
with torch.inference_mode():
    for b in CO_LO:
        x = torch.randn(b, 128)
        for _ in range(5):
            mo_hinh_xep_hang(x)            # khởi động
        lap = max(4, 2000 // b)
        mau_do = []
        for _ in range(7):
            t0 = time.perf_counter()
            for _ in range(lap):
                mo_hinh_xep_hang(x)
            mau_do.append((time.perf_counter() - t0) / lap)
        thoi_gian_lo[b] = float(np.median(mau_do))
for b in CO_LO:
    print(f"lô {b:>3}: {thoi_gian_lo[b] * 1e6:8.1f} µs cả lô · {thoi_gian_lo[b] / b * 1e6:7.2f} µs mỗi truy vấn")
do_tren_may("thời gian mỗi truy vấn ở lô 1", thoi_gian_lo[1] * 1e6, "µs")
do_tren_may("thời gian mỗi truy vấn ở lô 512", thoi_gian_lo[512] / 512 * 1e6, "µs")
do_tren_may("hệ số gom batch, lô 512 so với lô 1", thoi_gian_lo[1] / (thoi_gian_lo[512] / 512), "lần")
do_tren_may("thời gian cả lô 512 so với lô 1", thoi_gian_lo[512] / thoi_gian_lo[1], "lần")

Gom batch hạ nhu cầu phục vụ của mỗi truy vấn, vì một lần đọc trọng số được dùng cho nhiều truy vấn.
Nhưng cả lô chạy lâu hơn, chưa kể thời gian chờ cho đủ lô, nên đây là một đánh đổi với độ trễ chứ
không phải một khoản tiết kiệm miễn phí. Con số đo được là của CPU này và mô hình đồ chơi này, không
phải của GPU trong kịch bản, nên sổ tay không đưa nó vào sổ cái của sách.

Thay vào đó, ô dưới quét hai giả định trên chuỗi quy đổi của bảng 14: hệ số gom batch (nhu cầu phục vụ
chia cho bao nhiêu) và mức sử dụng (phần thời gian GPU đã trả tiền mà thật sự bận). Sổ cái của sách là
điểm hệ số 1, mức sử dụng 100 phần trăm. Mức giảm 20 phần trăm của sách ứng với hệ số 1.25.

In [ ]:
he_so = np.geomspace(1, 32, 60)
fig, ax = plt.subplots(figsize=(8.5, 4.2))
for k, u in enumerate((1.0, 0.6, 0.3)):  # giả định của sổ tay
    chi_phi = gio_gpu_moi_ngay(he_so_gom=he_so, muc_su_dung=u) * NGAY_MOI_NAM * GIA_GIO_SUY_LUAN * SO_NAM
    ax.plot(he_so, chi_phi / 1e6, color=MAU[k], lw=2,
            label=f"mức sử dụng {u:.0%}" + (" (như sổ cái của sách)" if u == 1.0 else " (giả định)"))
ax.scatter([1], [SL_TIEN / 1e6], color="black", zorder=5)
ax.annotate("sổ cái của sách", (1, SL_TIEN / 1e6), xytext=(6, 9), textcoords="offset points", fontsize=8)
ax.scatter([1.25], [SL_TIEN * (1 - GIAM_SL) / 1e6], color="black", marker="s", s=22, zorder=5)
ax.annotate("giảm 20 phần trăm", (1.25, SL_TIEN * (1 - GIAM_SL) / 1e6), xytext=(4, -15), textcoords="offset points",
            fontsize=8)
ax.set_xscale("log", base=2)
ax.set_yscale("log")
ax.set_xticks([1, 2, 4, 8, 16, 32])
ax.set_xticklabels(["1", "2", "4", "8", "16", "32"])
ax.set_yticks([0.05, 0.1, 0.2, 0.5, 1, 2, 5])
ax.set_yticklabels(["0.05", "0.1", "0.2", "0.5", "1", "2", "5"])
ax.minorticks_off()
ax.set_xlabel("hệ số gom batch: nhu cầu phục vụ mỗi truy vấn chia cho (thang log)")
ax.set_ylabel("chi phí suy luận ba năm (triệu $, thang log)")
ax.set_title("Cùng chuỗi quy đổi của bảng 14, đổi hai giả định", fontsize=10)
ax.legend(loc="upper right", fontsize=8)
ax.grid(alpha=0.3, which="both")
plt.tight_layout()
plt.show()

Hai giả định kéo ngược chiều nhau. Gom batch tốt chia chi phí theo đúng hệ số của nó; mức sử dụng thấp
nhân chi phí lên, vì GPU nhàn rỗi vẫn tính tiền. Một hệ thống gom batch giỏi nhưng phải giữ nhiều GPU
chờ giờ cao điểm có thể trả gần bằng sổ cái không gom batch. Đó là ý của câu sách: giảm độ trễ chỉ tiết
kiệm thời gian máy tăng tốc khi nó thật sự giảm nhu cầu phục vụ, thay vì dời việc hay dời thời gian
nhàn rỗi sang chỗ khác.

## 4. Carbon ở quy mô lớn, và carbon theo giờ

### Phép tính nhẩm 1.4: GPT-3

Một mô hình nền ở quy mô GPT-3 tiêu thụ 1,287 MWh điện khi huấn luyện, theo Patterson và cộng sự năm
2021. Kịch bản dùng cường độ carbon khoảng 0.429 kg mỗi kWh, và giả định mỗi xe chở khách phát thải
4.6 tấn CO2 một năm.

In [ ]:
NANG_LUONG_GPT3 = sach(1287, nguon="1,287 MWh · 0.429 kg mỗi kWh · 552,123 kg · 552 tấn · 4.6 tấn · 120 xe · 1 phần trăm · 1.2 xe") * 1000  # kWh
CUONG_DO_GPT3 = sach(0.429, nguon="1,287 MWh · 0.429 kg mỗi kWh · 552,123 kg · 552 tấn · 4.6 tấn · 120 xe · 1 phần trăm · 1.2 xe")
TAN_MOI_XE = sach(4.6, nguon="1,287 MWh · 0.429 kg mỗi kWh · 552,123 kg · 552 tấn · 4.6 tấn · 120 xe · 1 phần trăm · 1.2 xe")
MOT_PHAN_TRAM = sach(1, nguon="1,287 MWh · 0.429 kg mỗi kWh · 552,123 kg · 552 tấn · 4.6 tấn · 120 xe · 1 phần trăm · 1.2 xe") / 100
kg_gpt3 = NANG_LUONG_GPT3 * CUONG_DO_GPT3
theo_sach("phát thải khi huấn luyện (kg CO2)", kg_gpt3, sach=552123,
          nguon="1,287 MWh · 0.429 kg mỗi kWh · 552,123 kg · 552 tấn · 4.6 tấn · 120 xe · 1 phần trăm · 1.2 xe")
theo_sach("phát thải khi huấn luyện (tấn)", kg_gpt3 / 1000, sach=552,
          nguon="1,287 MWh · 0.429 kg mỗi kWh · 552,123 kg · 552 tấn · 4.6 tấn · 120 xe · 1 phần trăm · 1.2 xe")
theo_sach("tương đương số xe chạy một năm", kg_gpt3 / 1000 / TAN_MOI_XE, sach=120,
          nguon="1,287 MWh · 0.429 kg mỗi kWh · 552,123 kg · 552 tấn · 4.6 tấn · 120 xe · 1 phần trăm · 1.2 xe")
theo_sach("giảm 1 phần trăm năng lượng, số xe tương đương", MOT_PHAN_TRAM * kg_gpt3 / 1000 / TAN_MOI_XE, sach=1.2,
          nguon="1,287 MWh · 0.429 kg mỗi kWh · 552,123 kg · 552 tấn · 4.6 tấn · 120 xe · 1 phần trăm · 1.2 xe")

### Dự đoán

Cả hai phép tính nhẩm nhân năng lượng với một cường độ carbon duy nhất. Chương nói phát thải khi vận
hành còn phụ thuộc hiệu quả sử dụng điện (PUE) của cơ sở: ước lượng bậc một là năng lượng của phần
cứng, nhân PUE, nhân cường độ carbon. Sổ cái của bảng 16 cũng ghi rõ con số carbon của nó không gồm
phụ trội của cơ sở. Nếu cùng việc ấy chạy ở một lưới điện sạch hơn nhiều, hay ở một cơ sở có PUE kém,
con số thay đổi bao nhiêu lần? Các giá trị dưới đây là giả định của sổ tay, không phải số của sách.

In [ ]:
cuong_do = np.linspace(0.02, 0.9, 100)  # kg CO2 mỗi kWh, giả định
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.2))
ax1.plot(cuong_do, NANG_LUONG_GPT3 * cuong_do / 1000, color=MAU[0], lw=2)
ax1.scatter([CUONG_DO_GPT3], [kg_gpt3 / 1000], color="black", zorder=5)
ax1.annotate(f"phép tính nhẩm 1.4\n{kg_gpt3 / 1000:,.0f} tấn ≈ {kg_gpt3 / 1000 / TAN_MOI_XE:.0f} xe-năm",
             (CUONG_DO_GPT3, kg_gpt3 / 1000), xytext=(18, -40), textcoords="offset points", fontsize=8,
             arrowprops=dict(arrowstyle="-", color=XAM, lw=0.8))
ax1.set_xlabel("cường độ carbon của lưới điện (kg CO2 mỗi kWh)")
ax1.set_ylabel("phát thải khi huấn luyện (tấn CO2)")
ax1.set_title("1,287 MWh, đổi lưới điện", fontsize=10)
ax1.grid(alpha=0.3)

for k, pue in enumerate((1.0, 1.2, 1.5, 2.0)):  # giả định của sổ tay
    tan = SL_KG / 1000 * pue * cuong_do / CUONG_DO_CARBON
    ax2.plot(cuong_do, tan, color=MAU[k], lw=2,
             label="không tính cơ sở (như sổ cái)" if pue == 1.0 else f"PUE {pue:g} (giả định)")
ax2.scatter([CUONG_DO_CARBON], [SL_KG / 1000], color="black", zorder=5)
ax2.annotate("sổ cái của bảng 16", (CUONG_DO_CARBON, SL_KG / 1000), xytext=(12, -22), textcoords="offset points",
             fontsize=8, arrowprops=dict(arrowstyle="-", color=XAM, lw=0.8))
ax2.set_xlabel("cường độ carbon của lưới điện (kg CO2 mỗi kWh)")
ax2.set_ylabel("carbon suy luận ba năm (tấn CO2)")
ax2.set_title("Sổ cái suy luận, nhân thêm PUE", fontsize=10)
ax2.legend(loc="upper left", fontsize=8)
ax2.grid(alpha=0.3)
plt.tight_layout()
plt.show()
r_luoi = cuong_do.max() / cuong_do.min()
print(f"giữa lưới sạch nhất và bẩn nhất của phép quét: {r_luoi:.0f} lần · PUE 2 so với không tính cơ sở: 2 lần")

Trong các khoảng giả định của phép quét, cùng một lượng tính toán, chọn lưới điện đổi phát thải nhiều
lần hơn chọn PUE. Đó là lý
do chương xếp phần cứng, vùng đám mây và thời điểm chạy vào phạm vi của kỹ thuật có trách nhiệm, và
cũng là lý do chương cảnh báo: một phép tối ưu giảm được tổng chi phí sở hữu có thể không giảm năng
lượng hay phát thải, nếu nó đổi mức sử dụng, phần cứng hay nơi đặt tải.

### Lập lịch theo carbon: dời việc sang giờ sạch hơn

Chương nhắc tới việc lập lịch theo carbon, tức dời việc sang thời điểm hay vùng có lưới điện ít carbon
hơn. Ô dưới thử điều đó với một ngày giả định: cường độ carbon theo giờ cao vào buổi tối, thấp nhất vào
trưa nắng, và trũng thêm một lần lúc gần sáng. Một việc huấn luyện 8 GPU chạy 10 giờ (giả định), mỗi GPU 400 W như phép tính nhẩm 1.3,
được giao lúc 18 giờ và phải xong trong 24 giờ. So ba cách chạy.

In [ ]:
gio_dong_ho = (18 + np.arange(24)) % 24   # 24 giờ kể từ lúc giao việc
cuong_do_gio = (0.40 + 0.06 * np.cos(2 * np.pi * (gio_dong_ho - 19) / 24)   # giả định của sổ tay:
                - 0.17 * np.exp(-((gio_dong_ho - 13) / 2.2) ** 2)          # nắng trưa,
                - 0.12 * np.exp(-((gio_dong_ho - 3) / 1.8) ** 2)           # gió đêm,
                + 0.04 * np.exp(-((gio_dong_ho - 8) / 1.5) ** 2))          # cao điểm sáng
SO_GPU, SO_GIO = 8, 10                    # giả định của sổ tay
kwh_moi_gio = SO_GPU * CONG_SUAT_GPU
cua_so = np.array([cuong_do_gio[i:i + SO_GIO].sum() for i in range(24 - SO_GIO + 1)])
bat_dau_tot = int(cua_so.argmin())
cach = {
    "chạy ngay": np.arange(SO_GIO),
    "cửa sổ liền 10 giờ sạch nhất": np.arange(bat_dau_tot, bat_dau_tot + SO_GIO),
    "10 giờ sạch nhất, tạm dừng được": np.sort(np.argsort(cuong_do_gio)[:SO_GIO]),
}
kg_cach = {}
for ten, gio in cach.items():
    kg_cach[ten] = kwh_moi_gio * cuong_do_gio[gio].sum()
    print(f"{ten:>32}: bắt đầu {gio_dong_ho[gio[0]]:02d}:00 · {kg_cach[ten]:6.1f} kg CO2 "
          f"({kg_cach[ten] / kg_cach['chạy ngay'] - 1:+.0%} so với chạy ngay)")
ten_cach = list(cach)
print(f"tạm dừng được có ít carbon hơn cửa sổ liền không: "
      f"{'có' if kg_cach[ten_cach[2]] < kg_cach[ten_cach[1]] - 1e-9 else 'không, mười giờ sạch nhất đã liền nhau'}")

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 4.8), sharex=True, gridspec_kw=dict(height_ratios=[2, 1.1]))
x = np.arange(24)
ax1.step(np.arange(25), np.r_[cuong_do_gio, cuong_do_gio[-1]], where="post", color=MAU[0], lw=2)
ax1.set_ylabel("kg CO2 mỗi kWh\n(giả định)")
ax1.set_ylim(0, cuong_do_gio.max() * 1.15)
ax1.set_title("Một ngày giả định của lưới điện, và ba cách xếp cùng một việc", fontsize=10)
ax1.grid(alpha=0.3)
for k, (ten, gio) in enumerate(cach.items()):
    ax2.broken_barh([(g, 1) for g in gio], (k - 0.35, 0.7), color=MAU[k + 1])
    ax2.text(23.8, k, f"{kg_cach[ten]:.1f} kg", va="center", ha="right", fontsize=8)
ax2.set_yticks(range(len(cach)))
ax2.set_yticklabels(list(cach), fontsize=8)
ax2.set_ylim(len(cach) - 0.5, -0.5)
ax2.set_xticks(x[::2])
ax2.set_xticklabels([f"{h:02d}h" for h in gio_dong_ho[::2]], fontsize=8)
ax2.set_xlim(0, 24)
ax2.set_xlabel("giờ trong ngày, kể từ lúc giao việc")
plt.tight_layout()
plt.show()

Cùng một việc, cùng số kWh, chỉ đổi giờ chạy. Cách nào cũng tốn đúng ngần ấy năng lượng, nên phần hoá
đơn tính theo kWh không đổi, nhưng carbon thì có. Khi lưới điện có hai vùng trũng, một việc tạm dừng
được ăn được cả hai, còn một cửa sổ liền phải chọn một. Sự linh hoạt ấy là một yêu cầu thiết kế: lưu
trạng thái huấn luyện đều đặn để dừng rồi chạy tiếp, và một hàng đợi biết hạn chót. Con số cụ thể ở đây đến từ đường cong giả định; với lưới điện thật, phải đọc cường độ
carbon của vùng theo giờ.

## 5. Dấu vết dữ liệu: quản trị bằng phép tính

Quản trị dữ liệu là lớp thực thi làm cho trách nhiệm giải trình có thể có được. Chương mở mục này bằng
một quyết định: tháng 1 năm 2023, Uỷ ban Bảo vệ Dữ liệu Ireland phạt Meta Ireland hai khoản riêng, vì
dựa sai vào căn cứ hợp đồng để xử lý dữ liệu cho quảng cáo cá nhân hoá, cùng các vi phạm về minh bạch
và công bằng, chứ không phải vì rò rỉ dữ liệu.

In [ ]:
PHAT = [sach(210, nguon="tháng 1 năm 2023 · 210 triệu · 180 triệu · 390 triệu euro"),
        sach(180, nguon="tháng 1 năm 2023 · 210 triệu · 180 triệu · 390 triệu euro")]  # triệu euro
theo_sach("tổng hai khoản phạt (triệu euro)", sum(PHAT), sach=390,
          nguon="tháng 1 năm 2023 · 210 triệu · 180 triệu · 390 triệu euro")

Phần còn lại của mục là kiến trúc: kiểm soát truy cập, quyền riêng tư, tuân thủ, phả hệ và kiểm toán.
Phần lớn là thiết kế chứ không phải phép tính. Ba chỗ tính được thì sổ tay tính.

### Một yêu cầu xoá đi tới đâu: đồ thị phả hệ

Khi một người dùng thực hiện quyền được xoá, sách viết rằng đồ thị phả hệ dữ liệu xác định các artifact
dẫn xuất ứng viên, như đặc trưng, embedding và các phiên bản mô hình đã huấn luyện, để đưa vào quy
trình phù hợp: xoá, vô hiệu hoá, hạn chế, học máy quên đi, huấn luyện lại, hay xem xét pháp lý. Ô dưới
dựng một đồ thị nhỏ cho hệ thống phát hiện từ khoá (KWS) của chương (đồ thị là giả định của sổ tay) và
đi theo các cạnh từ bản thu của một người.

In [ ]:
PHA_HE = {  # nguồn → các artifact dẫn xuất trực tiếp từ nó
    "bản thu: người dùng 17": ["đặc trưng: lô tháng 3"],
    "bản thu: người dùng 42": ["đặc trưng: lô tháng 3"],
    "bản thu: người dùng 99": ["đặc trưng: lô tháng 4"],
    "đặc trưng: lô tháng 2": ["tập huấn luyện v2"],
    "đặc trưng: lô tháng 3": ["tập huấn luyện v3", "embedding giọng: v1"],
    "đặc trưng: lô tháng 4": ["tập huấn luyện v4"],
    "tập huấn luyện v2": ["mô hình v2"],
    "tập huấn luyện v3": ["mô hình v3", "tập huấn luyện v4"],
    "tập huấn luyện v4": ["mô hình v4"],
    "mô hình v3": ["mô hình v4"],                   # v4 tinh chỉnh từ v3
    "mô hình v4": ["mô hình v4, lượng tử hóa", "triển khai: dịch vụ máy chủ"],
    "mô hình v4, lượng tử hóa": ["triển khai: bản cập nhật thiết bị tháng 5"],
    "embedding giọng: v1": ["triển khai: dịch vụ máy chủ"],
}
QUY_TRINH = {  # loại artifact → quy trình ứng viên; quyết định cụ thể tuỳ trường hợp
    "bản thu": "xoá", "đặc trưng": "xoá hoặc tính lại", "embedding giọng": "vô hiệu hoá, tính lại",
    "tập huấn luyện": "dựng lại không có bản ghi", "mô hình": "huấn luyện lại, học máy quên đi, hay xem xét pháp lý",
    "triển khai": "theo quyết định của mô hình nguồn",
}


def bi_anh_huong(nguon):
    """Mọi artifact đi tới được từ `nguon` theo cạnh của đồ thị phả hệ, theo thứ tự gặp."""
    da_gap, hang_doi = [], [nguon]
    while hang_doi:
        nut = hang_doi.pop(0)
        for con in PHA_HE.get(nut, []):
            if con not in da_gap:
                da_gap.append(con)
                hang_doi.append(con)
    return da_gap


tat_ca = set(PHA_HE) | {c for v in PHA_HE.values() for c in v}
for nguoi in ("bản thu: người dùng 42", "bản thu: người dùng 99"):
    ds = bi_anh_huong(nguoi)
    print(f"{nguoi}: {len(ds)} artifact dẫn xuất trong {len(tat_ca)} nút của đồ thị")
    for a in ds:
        loai = next(k for k in QUY_TRINH if a.startswith(k))
        print(f"   {a:<44} → {QUY_TRINH[loai]}")
assert "mô hình v2" not in bi_anh_huong("bản thu: người dùng 42")
print("mô hình v2 không nằm trong phạm vi: nó học từ lô tháng 2, trước khi bản thu của người dùng 42 có mặt")

Xoá một bản thu trong kho chạm tới một nút. Phạm vi thật của yêu cầu là cả chuỗi dẫn xuất, gồm cả mô
hình tinh chỉnh và bản lượng tử hóa đã nằm trên thiết bị của người khác. Không có đồ thị, câu hỏi
"mô hình nào đã học từ bản ghi này" trở thành một cuộc điều tra thủ công. Có đồ thị, nó là một truy vấn.

### Trọng số có nhớ không: suy luận thành viên

Một ngộ nhận chương bác bỏ là cho rằng trọng số mô hình nằm ngoài quản trị dữ liệu: dữ liệu đã được
biên dịch vào trọng số thì coi như đã mất. Sách viết rằng mô hình có thể ghi nhớ dữ liệu huấn luyện, và
tấn công suy luận thành viên có thể cho thấy một bản ghi có nằm trong tập huấn luyện hay không, theo
Shokri năm 2017.

Thí nghiệm dưới đây dùng 1,000 ảnh FashionMNIST để huấn luyện một mạng nhỏ. Ta cài thêm 50 "bản ghi
riêng": ảnh mang một nhãn sai ngẫu nhiên, thứ mô hình chỉ có thể khớp bằng cách ghi nhớ. Chúng đóng vai
những bản ghi không theo khuôn mẫu chung của tập dữ liệu, như dữ liệu của một người hiếm gặp. Kẻ tấn
công chỉ xem mất mát của mô hình trên một bản ghi: mất mát càng thấp, càng đoán là thành viên. Ta đo
AUC của phép đoán ấy, với 0.5 là đoán ngẫu nhiên.

### Dự đoán

Kẻ tấn công nhận ra bản ghi thường hay bản ghi riêng dễ hơn? Và nếu ta xoá 50 bản ghi riêng khỏi kho
nhưng giữ nguyên mô hình, AUC trên chúng còn bao nhiêu?

In [ ]:
tap = torchvision.datasets.FashionMNIST(DATA, train=True, download=True)
X_anh = tap.data.reshape(-1, 784).float() / 255.0
nhan_goc = tap.targets.clone()
r_mi = np.random.default_rng(SEED)
thu_tu = r_mi.permutation(len(X_anh))
N_THUONG, N_RIENG = 1000, 50
thuong_tv, thuong_ngoai = thu_tu[:N_THUONG], thu_tu[N_THUONG:2 * N_THUONG]
rieng_tv = thu_tu[2 * N_THUONG:2 * N_THUONG + N_RIENG]
rieng_ngoai = thu_tu[2 * N_THUONG + N_RIENG:2 * N_THUONG + 2 * N_RIENG]
nhan = nhan_goc.clone()
for nhom in (rieng_tv, rieng_ngoai):  # nhãn sai ngẫu nhiên, khác nhãn thật
    nhan[nhom] = torch.tensor((nhan_goc[nhom].numpy() + r_mi.integers(1, 10, len(nhom))) % 10)
print(f"thành viên: {N_THUONG} ảnh thường + {N_RIENG} bản ghi riêng · ngoài tập: cũng {N_THUONG} + {N_RIENG}")

In [ ]:
def huan_luyen(chi_so, suy_giam=0.0, so_vong=100, bo_bot=0.0):
    """Một mạng 784-256-10, Adam theo lô 100. Trả về mô hình ở chế độ đánh giá."""
    dat_hat_giong()
    chi_so = torch.tensor(chi_so)
    m = torch.nn.Sequential(torch.nn.Linear(784, 256), torch.nn.ReLU(), torch.nn.Dropout(bo_bot),
                            torch.nn.Linear(256, 10))
    toi_uu = torch.optim.AdamW(m.parameters(), lr=1e-3, weight_decay=suy_giam)
    g = torch.Generator().manual_seed(SEED)
    for _ in range(so_vong):
        m.train()
        p = torch.randperm(len(chi_so), generator=g)
        for b in range(0, len(chi_so), 100):
            j = chi_so[p[b:b + 100]]
            toi_uu.zero_grad()
            torch.nn.functional.cross_entropy(m(X_anh[j]), nhan[j]).backward()
            toi_uu.step()
    return m.eval()


def mat_mat(m, chi_so):
    with torch.no_grad():
        return torch.nn.functional.cross_entropy(m(X_anh[chi_so]), nhan[chi_so], reduction="none").numpy()


def auc_thanh_vien(m, trong, ngoai):
    """AUC của phép đoán 'mất mát thấp nghĩa là thành viên'."""
    a, b = mat_mat(m, trong), mat_mat(m, ngoai)
    return roc_auc_score(np.r_[np.ones(len(a)), np.zeros(len(b))], -np.r_[a, b])


def do_chinh_xac(m, chi_so):
    with torch.no_grad():
        return (m(X_anh[chi_so]).argmax(1) == nhan_goc[chi_so]).float().mean().item()


t0 = time.perf_counter()
thanh_vien = np.r_[thuong_tv, rieng_tv]
mh_qua_khop = huan_luyen(thanh_vien)                                    # 100 vòng, không điều chuẩn
mh_dieu_chuan = huan_luyen(thanh_vien, suy_giam=0.1, so_vong=30, bo_bot=0.5)
mh_huan_luyen_lai = huan_luyen(thuong_tv)                               # sau yêu cầu xoá: không có 50 bản ghi riêng
do_tren_may("thời gian huấn luyện ba mô hình", time.perf_counter() - t0, "s")
AUC = {
    "ảnh thường, mô hình quá khớp": auc_thanh_vien(mh_qua_khop, thuong_tv, thuong_ngoai),
    "bản ghi riêng, mô hình quá khớp": auc_thanh_vien(mh_qua_khop, rieng_tv, rieng_ngoai),
    "bản ghi riêng, có điều chuẩn": auc_thanh_vien(mh_dieu_chuan, rieng_tv, rieng_ngoai),
    "bản ghi riêng, xoá khỏi kho, giữ mô hình": auc_thanh_vien(mh_qua_khop, rieng_tv, rieng_ngoai),
    "bản ghi riêng, huấn luyện lại không có chúng": auc_thanh_vien(mh_huan_luyen_lai, rieng_tv, rieng_ngoai),
}
for ten, m in (("quá khớp", mh_qua_khop), ("có điều chuẩn", mh_dieu_chuan), ("huấn luyện lại", mh_huan_luyen_lai)):
    do_tren_may(f"độ chính xác trên ảnh ngoài tập, mô hình {ten}", 100 * do_chinh_xac(m, thuong_ngoai), "%")
for ten, v in AUC.items():
    do_tren_may(f"AUC suy luận thành viên, {ten}", v)

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.2), gridspec_kw=dict(width_ratios=[1, 1.25]))
bins = np.linspace(-3, 1.5, 28)
for nhom, ten, mau in ((rieng_tv, "bản ghi riêng đã huấn luyện", MAU[1]), (rieng_ngoai, "bản ghi riêng ngoài tập", MAU[0])):
    ax1.hist(np.clip(np.log10(mat_mat(mh_qua_khop, nhom) + 1e-12), -3, 1.5), bins=bins, color=mau, alpha=0.6, label=ten)
ax1.set_xlabel("log10 của mất mát trên bản ghi")
ax1.set_ylabel("số bản ghi")
ax1.set_title("Mô hình quá khớp: mất mát của 100 bản ghi riêng", fontsize=10)
ax1.legend(loc="upper left", fontsize=8)

ten_auc = list(AUC)
y = np.arange(len(ten_auc))
mau_auc = [MAU[0], MAU[1], MAU[3], MAU[1], MAU[2]]
ax2.barh(y, [AUC[k] for k in ten_auc], color=mau_auc, height=0.6)
for yi, k in zip(y, ten_auc):
    ax2.text(AUC[k] + 0.01, yi, f"{AUC[k]:.2f}", va="center", fontsize=8)
ax2.axvline(0.5, color=XAM, ls="--", lw=1)
ax2.text(0.51, len(ten_auc) - 1, "← đoán ngẫu nhiên", fontsize=8, color=XAM, va="center")
ax2.set_yticks(y)
ax2.set_yticklabels(ten_auc, fontsize=8)
ax2.invert_yaxis()
ax2.set_xlim(0.3, 1.08)
ax2.set_xlabel("AUC của suy luận thành viên (đo trên máy này)")
ax2.set_title("Kẻ tấn công nhận ra thành viên tới đâu", fontsize=10)
plt.tight_layout()
plt.show()

Ở ảnh thường, phép đoán tốt hơn ngẫu nhiên nhưng không nhiều, vì mô hình khớp chúng bằng khuôn mẫu chung,
và ảnh ngoài tập cũng theo khuôn mẫu ấy. Bản ghi riêng thì khác: mô hình quá khớp chỉ khớp được chúng
bằng cách ghi nhớ, và hình bên trái cho thấy hai nhóm mất mát tách xa nhau. Điều chuẩn làm khó kẻ tấn
công hơn nhưng không xoá được dấu vết.

Hai thanh cuối là câu hỏi quản trị. Xoá bản ghi khỏi kho không đổi một trọng số nào, nên AUC giữ nguyên:
dấu vết vẫn nằm trong mô hình đang phục vụ. Chỉ khi huấn luyện lại không có chúng, AUC mới về quanh mức
đoán ngẫu nhiên. Huấn luyện lại từ đầu là mốc chuẩn mà học máy quên đi cố gắng đạt với chi phí thấp
hơn. Sách thận trọng ở hai chỗ, và sổ tay giữ cả hai: trọng số cụ thể có phải là dữ liệu cá nhân hay
không là câu hỏi tuỳ trường hợp, cần chuyên gia pháp lý và quyền riêng tư; và suy luận thành viên có thể
bổ sung cho các phép kiểm tra, nhưng không xác lập được một bảo đảm hình thức.

### Quyền riêng tư vi sai: một ngân sách tiêu dần

Bảo đảm hình thức ấy là quyền riêng tư vi sai. Phần C của bài thực hành đi kèm chương dùng cơ chế đơn
giản nhất: công bố trung bình của một đại lượng bị chặn. Mỗi giá trị được cắt vào một khoảng công khai
$[a, b]$, nên một người chỉ đổi được trung bình của $n$ người tối đa $(b - a)/n$, gọi là độ nhạy. Cơ chế
cộng nhiễu Laplace có thang đo bằng độ nhạy chia $\varepsilon$, nên sai số tuyệt đối kỳ vọng của một lần
công bố đúng bằng thang đo ấy. Theo phép hợp thành tuần tự cơ bản, $k$ lần công bố, mỗi lần tiêu
$\varepsilon$, tiêu tổng cộng $k\varepsilon$. Mọi giá trị dưới đây là giả định của sổ tay.

In [ ]:
CAN_DUOI, CAN_TREN = 0.0, 50.0   # giả định: số lần thiết bị bị đánh thức mỗi ngày, cắt vào [0, 50]
r_dp = np.random.default_rng(SEED)


def cong_bo_trung_binh(x, eps, rng, so_lan=1):
    """`so_lan` lần công bố trung bình bị chặn bằng cơ chế Laplace, mỗi lần tiêu `eps`."""
    x = np.clip(x, CAN_DUOI, CAN_TREN)
    do_nhay = (CAN_TREN - CAN_DUOI) / len(x)
    return x.mean() + rng.laplace(0.0, do_nhay / eps, size=so_lan)


eps_quet = np.geomspace(0.05, 5, 25)
sai_so = {}
for n in (100, 1000, 10000):    # giả định: số người dùng góp vào trung bình
    x = r_dp.gamma(2.0, 6.0, n)
    sai_so[n] = [np.abs(cong_bo_trung_binh(x, e, r_dp, so_lan=4000) - np.clip(x, 0, 50).mean()).mean()
                 for e in eps_quet]
    print(f"n = {n:>6}: độ nhạy {(CAN_TREN - CAN_DUOI) / n:.4f} · ở ε = 1, sai số mô phỏng "
          f"{np.interp(1.0, eps_quet, sai_so[n]):.4f}, lý thuyết {(CAN_TREN - CAN_DUOI) / n:.4f}")

NGAN_SACH = 1.0                 # giả định: tổng ε được phép tiêu
x = r_dp.gamma(2.0, 6.0, 1000)
gia_tri_that = np.clip(x, 0, 50).mean()
so_lan_quet = [1, 2, 4, 8, 16, 32]
sai_mot_lan, sai_trung_binh = [], []
for k in so_lan_quet:
    mau_ = cong_bo_trung_binh(x, NGAN_SACH / k, r_dp, so_lan=3000 * k).reshape(3000, k)
    sai_mot_lan.append(np.abs(mau_[:, 0] - gia_tri_that).mean())
    sai_trung_binh.append(np.abs(mau_.mean(axis=1) - gia_tri_that).mean())
    print(f"chia ε tổng {NGAN_SACH:g} cho {k:>2} lần công bố: sai số một lần {sai_mot_lan[-1]:.3f} · "
          f"trung bình cả {k} lần {sai_trung_binh[-1]:.3f}")

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.2))
for k, n in enumerate(sai_so):
    ax1.plot(eps_quet, sai_so[n], "o", ms=3, color=MAU[k], label=f"{n:,} người, mô phỏng")
    ax1.plot(eps_quet, (CAN_TREN - CAN_DUOI) / n / eps_quet, color=MAU[k], lw=1, alpha=0.7)
ax1.plot([], [], color="0.4", lw=1, label="lý thuyết: độ nhạy ÷ ε")
ax1.set_xscale("log")
ax1.set_yscale("log")
ax1.set_xlabel("ε của một lần công bố (thang log)")
ax1.set_ylabel("sai số tuyệt đối trung bình (lần đánh thức)")
ax1.set_title("Một lần công bố: ε nhỏ hơn, nhiễu lớn hơn", fontsize=10)
ax1.legend(loc="lower left", fontsize=8)
ax1.grid(alpha=0.3, which="both")

ax2.plot(so_lan_quet, sai_mot_lan, "o-", color=MAU[1], lw=2, label="một lần công bố bất kỳ")
ax2.plot(so_lan_quet, sai_trung_binh, "s-", color=MAU[2], lw=2, label="trung bình của cả k lần")
ax2.set_xscale("log", base=2)
ax2.set_yscale("log")
ax2.set_xticks(so_lan_quet)
ax2.set_xticklabels([str(k) for k in so_lan_quet])
ax2.set_xlabel(f"số lần công bố k, chia đều ε tổng {NGAN_SACH:g} (thang log)")
ax2.set_ylabel("sai số tuyệt đối trung bình (lần đánh thức)")
ax2.set_title("Cùng một ngân sách, chia cho nhiều lần công bố", fontsize=10)
ax2.legend(loc="upper left", fontsize=8)
ax2.grid(alpha=0.3, which="both")
plt.tight_layout()
plt.show()

Bên trái, sai số mô phỏng nằm đúng trên đường lý thuyết: gấp đôi số người thì độ nhạy giảm một nửa, giảm
$\varepsilon$ một nửa thì nhiễu gấp đôi. Bên phải là lý do một bộ theo dõi ngân sách là bắt buộc chứ
không phải tuỳ chọn. Cùng một tổng $\varepsilon$, chia cho nhiều lần công bố thì mỗi lần nhiễu hơn theo
đúng số lần. Gộp cả $k$ lần lại cũng không lấy lại được độ chính xác của một lần công bố duy nhất:
trung bình của chúng vẫn nhiễu gấp khoảng căn bậc hai của $k$ lần so với dồn cả ngân sách vào một lần công
bố. Bài thực hành nói gọn: giữ dữ liệu lâu hay ngắn đổi dung lượng lưu
trữ, còn $\varepsilon$ chỉ do cơ chế và phép hợp thành quyết định.

## Thử thêm

1. Ở phần 1, đổi `TAN_SUAT` thành 10 (mười lần suy luận mỗi giây). Mô hình nào còn qua ngân sách năng
   lượng một ngày của thiết bị đeo?
2. Ở phần 2, đổi `CHU_KY_MOI_NAM` thành 12 (huấn luyện lại mỗi tháng) rồi chạy lại các ô sau đó. Tỉ lệ
   suy luận trên huấn luyện còn bao nhiêu, và câu trả lời cho câu hỏi tự kiểm tra có đổi chiều không?
   Các dòng ✗ khi ấy là điều được chờ đợi.
3. Ở phần 4, đổi `SO_GIO` thành 4 hay 20. Lập lịch theo carbon có lợi nhiều nhất cho việc dài hay việc
   ngắn, khi hạn chót vẫn là 24 giờ?
4. Ở phần 5, tăng `N_RIENG` lên 200, hoặc huấn luyện mô hình quá khớp với `so_vong=20`. AUC của bản ghi
   riêng đổi thế nào?

## Tóm lại

* Ở biên, "vừa hay không" có hai điều kiện, công suất và độ trễ, và mỗi lần "không" có lý do riêng; ngân
  sách năng lượng của pin là câu hỏi thứ ba khi mô hình không chạy liên tục.
* Một giờ GPU 400 W ở 0.4 kg mỗi kWh là 0.16 kg CO2. Với hệ số ấy, ba sổ cái của kịch bản khép lại đúng
  như bảng 13 đến 16: suy luận 73.5 phần trăm, vận hành 24.6, huấn luyện 1.9, tỉ lệ 40 trên 1, và carbon
  suy luận gấp khoảng 63 lần huấn luyện.
* Vì thế giảm 20 phần trăm nhu cầu phục vụ của mỗi truy vấn tiết kiệm khoảng 304 nghìn đô la và 19 tấn
  CO2, nhiều hơn hẳn việc giảm một nửa thời gian huấn luyện; nhưng gom batch chỉ tiết kiệm khi mức sử
  dụng theo kịp.
* Cùng 1,287 MWh, lưới điện và PUE đổi phát thải nhiều lần; dời một việc sang giờ sạch hơn giảm carbon
  mà không giảm kWh.
* Một yêu cầu xoá đi theo đồ thị phả hệ tới mọi artifact dẫn xuất. Xoá khỏi kho không xoá dấu vết trong
  trọng số; huấn luyện lại thì có. Quyền riêng tư vi sai là một ngân sách: mỗi lần công bố tiêu một phần.

---
### Nguồn và giấy phép

Sổ tay này đi kèm bài học [Kỹ thuật có trách nhiệm — viết lại những gì đặc tả bỏ sót](https://d3lu8vk4we0dc.cloudfront.net/lessons/vol1-ch15-responsible-engr/), dựng từ chương
[*Responsible Engineering*](https://mlsysbook.ai/vol1/responsible_engr/responsible_engr.html) của sách. Một phần sổ tay phỏng theo phòng thí nghiệm `labs/vol1/lab_15_responsible_engr.py` của sách (© 2026 President and Fellows of Harvard College, CC BY-NC-SA 4.0).

Nội dung gốc thuộc Machine Learning Systems của Vijay Janapa Reddi và cộng sự, MIT Press. Bản quyền © 2024-2026 Harvard University. Giấy phép CC BY-NC-SA 4.0. Bản tiếng Việt này là tác phẩm phái sinh dùng cùng giấy phép: ghi công, chia sẻ tương tự, phi thương mại.